# 🏦 Loan Approval Model — Classification & Cost-Sensitive Decision Making

**Session 4 | Case Study 1 | DSA & ML for Business**

---

### Business Context
- Global consumer lending: **$4.2 Trillion** outstanding — 2-5% default rate
- **1% improvement** in prediction = **billions** in saved write-offs
- Regulatory requirement: models must be **explainable**
- Challenge: **class imbalance** — only 5-8% of loans actually default
- **False Negative cost >> False Positive cost** — missing a default is catastrophic

### What You'll Learn
1. Handle **imbalanced classification** with class weights
2. Train **Decision Tree** (explainable) and **Random Forest** (accurate)
3. Evaluate with **precision, recall, F1, ROC-AUC**
4. Apply **business cost matrix** to select the optimal model
5. **ROC threshold optimization** for minimum total business cost

## Step 1: Import Libraries & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve,
                             roc_auc_score, f1_score)
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("../datasets/loan_approval.csv")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nClass distribution:")
print(df['default'].value_counts())
print(f"\nDefault rate: {df['default'].mean():.2%}")
df.describe().round(2)

## Step 2: EDA — Feature Analysis & Class Imbalance

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
features = ['credit_score', 'income', 'loan_amount', 'employment_years', 'debt_to_income', 'previous_defaults']
for ax, feat in zip(axes.flat, features):
    for label, color in [(0, '#10B981'), (1, '#EF4444')]:
        ax.hist(df[df['default'] == label][feat], bins=30, alpha=0.5,
                label=f'{"Default" if label else "No Default"}', color=color, density=True)
    ax.set_title(feat.replace('_', ' ').title(), fontweight='bold')
    ax.legend(fontsize=8)
plt.suptitle('Feature Distributions by Default Status', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 3: Train-Test Split

In [ ]:
X = df.drop('default', axis=1)
y = df['default']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Training set: {X_train.shape[0]} samples | Default rate: {y_train.mean():.2%}")
print(f"Test set:     {X_test.shape[0]} samples | Default rate: {y_test.mean():.2%}")

## Step 4: Train Models — Decision Tree, Random Forest, SGD

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'SGD Classifier': SGDClassifier(loss='log_loss', class_weight='balanced', random_state=42, max_iter=1000),
}
results = {}
for name, model in models.items():
    if 'SGD' in name:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.decision_function(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
    results[name] = {'model': model, 'y_pred': y_pred, 'y_prob': y_prob,
                     'roc_auc': roc_auc_score(y_test, y_prob)}
    print(f"\n{'='*50}")
    print(f"{name} | ROC-AUC: {results[name]['roc_auc']:.4f}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))

## Step 5: Decision Tree Visualization & Feature Importance

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(results['Decision Tree']['model'], feature_names=X.columns,
          class_names=['No Default', 'Default'], filled=True, rounded=True, ax=ax, fontsize=9, proportion=True)
ax.set_title('Decision Tree — Human-Readable Credit Risk Rules', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, name in zip(axes, ['Decision Tree', 'Random Forest']):
    model = results[name]['model']
    importances = pd.Series(model.feature_importances_, index=X.columns).sort_values()
    importances.plot(kind='barh', ax=ax, color='#2563EB')
    ax.set_title(f'{name} — Feature Importance', fontweight='bold')
    ax.set_xlabel('Importance'); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## Step 6: Business Cost Matrix & ROC Threshold Optimization

**Cost Matrix:**
- False Negative (miss a default): **$50,000**
- False Positive (reject a good borrower): **$5,000**

In [ ]:
COST_FN = 50000; COST_FP = 5000
print(f"Cost of missed default (FN): ${COST_FN:,}")
print(f"Cost of rejecting good borrower (FP): ${COST_FP:,}")
print(f"FN/FP cost ratio: {COST_FN/COST_FP:.0f}x")

for name, res in results.items():
    cm = confusion_matrix(y_test, res['y_pred'])
    tn, fp, fn, tp = cm.ravel()
    total_cost = fn * COST_FN + fp * COST_FP
    print(f"\n{name}: TP={tp} FP={fp} FN={fn} TN={tn} | Total Cost: ${total_cost:,.0f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ax = axes[0]
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={res['roc_auc']:.3f})", linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curves', fontweight='bold'); ax.legend(); ax.grid(True, alpha=0.3)

rf_prob = results['Random Forest']['y_prob']
thresholds_range = np.arange(0.1, 0.9, 0.01)
total_costs = []
for thresh in thresholds_range:
    y_pred_thresh = (rf_prob >= thresh).astype(int)
    cm = confusion_matrix(y_test, y_pred_thresh)
    tn, fp, fn, tp = cm.ravel()
    total_costs.append(fn * COST_FN + fp * COST_FP)
optimal_idx = np.argmin(total_costs)
optimal_threshold = thresholds_range[optimal_idx]
min_cost = total_costs[optimal_idx]

axes[1].plot(thresholds_range, total_costs, 'b-', linewidth=2)
axes[1].axvline(x=optimal_threshold, color='red', linestyle='--', label=f'Optimal: {optimal_threshold:.2f}')
axes[1].scatter([optimal_threshold], [min_cost], color='red', s=150, zorder=5)
axes[1].set_xlabel('Threshold'); axes[1].set_ylabel('Total Business Cost ($)')
axes[1].set_title('Cost-Optimized Threshold (Random Forest)', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f"\n✅ Optimal threshold: {optimal_threshold:.2f} | Min cost: ${min_cost:,.0f}")

## Step 7: Strategic Analysis — Board-Ready Recommendation

In [ ]:
y_pred_optimal = (rf_prob >= optimal_threshold).astype(int)
cm_optimal = confusion_matrix(y_test, y_pred_optimal)
tn, fp, fn, tp = cm_optimal.ravel()

print("=" * 70)
print("MODEL DEPLOYMENT RECOMMENDATION — Bank Risk Committee")
print("=" * 70)
print(f"\n1. RECOMMENDED MODEL: Random Forest (threshold={optimal_threshold:.2f})")
print(f"   Recall: {tp/(tp+fn):.2%} | Precision: {tp/(tp+fp):.2%}")
print(f"   ROC-AUC: {results['Random Forest']['roc_auc']:.4f}")

annual_loans = 50000
default_rate = y_test.mean()
total_defaults = annual_loans * default_rate
print(f"\n2. FINANCIAL IMPACT (at {annual_loans:,} annual loans):")
print(f"   Expected defaults: {total_defaults:,.0f}")
print(f"   Value of 1% recall improvement: ${total_defaults * 0.01 * COST_FN:,.0f}/year")

print(f"\n3. TOP RISK FACTORS:")
rf_importance = pd.Series(results['Random Forest']['model'].feature_importances_, index=X.columns).sort_values(ascending=False)
for feat, imp in rf_importance.head(3).items():
    print(f"   {feat}: {imp:.3f}")